# 01. KITTI 데이터 준비

KITTI를 다운로드하고 YOLO 형식으로 바꿔요.

- 클래스: Van→Car, Person_sitting→Pedestrian 으로 합쳐서 **Car / Pedestrian / Cyclist** 3개만 써요.
- `eval_val.txt` 1,000장은 **학습에서 빠져요** (시험지).
- 학습 중 점검용(dev)은 나머지 사진의 10%를 따로 떼서 써요.

Colab은 세션이 끝나면 `/content`가 지워져요. 새로 접속할 때마다 이 노트북을 다시 실행하면 돼요 (약 10~15분).

## 1. 저장소 받기

In [ ]:
!git clone -q https://github.com/gyu-young-04/embeded_ondeviceAI.git /content/embeded_ondeviceAI || git -C /content/embeded_ondeviceAI pull -q
%cd /content/embeded_ondeviceAI

## 2. KITTI 다운로드 (이미지 약 12GB + 라벨 16MB)

공식 사이트(cvlibs.net) 다운로드 링크와 같은 파일이에요. 링크가 안 되면 사이트에서 회원가입 후 받은 링크로 바꿔 주세요.
test 이미지는 필요 없어서 `training/` 폴더만 풀어요.

In [ ]:
%%bash
set -e   # 중간에 실패하면 바로 멈추고 어디서 실패했는지 보여 줌
mkdir -p /content/kitti && cd /content/kitti
for f in data_object_label_2.zip data_object_image_2.zip; do
  wget -c -q --show-progress https://s3.eu-central-1.amazonaws.com/avg-kitti/$f
  unzip -q -n $f 'training/*'
done
[ "$(ls training/image_2 | wc -l)" = 7481 ] || { echo '사진이 7481장이 아니에요. /content/kitti 를 지우고 다시 실행하세요.'; exit 1; }
ls training/image_2 | wc -l   # 7481 이 나와야 해요

## 3. YOLO 형식으로 변환 + train / dev / eval 나누기

In [ ]:
!pip -q install pillow
!python scripts/prepare_kitti.py --kitti /content/kitti --out /content/kitti_yolo

## 4. 눈으로 확인하기

네모가 차·사람·자전거에 제대로 쳐져 있는지 봐요.

In [ ]:
import random
from pathlib import Path
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt

root = Path('/content/kitti_yolo')
names = ['Car', 'Pedestrian', 'Cyclist']
colors = ['red', 'lime', 'cyan']
ids = [Path(p).stem for p in (root / 'train.txt').read_text().split()]

fig, axes = plt.subplots(3, 1, figsize=(14, 9))
for ax, image_id in zip(axes, random.sample(ids, 3)):
    img = Image.open(root / 'images' / f'{image_id}.png').convert('RGB')
    draw = ImageDraw.Draw(img)
    W, H = img.size
    for line in (root / 'labels' / f'{image_id}.txt').read_text().splitlines():
        c, cx, cy, w, h = line.split()
        cx, cy, w, h = float(cx) * W, float(cy) * H, float(w) * W, float(h) * H
        draw.rectangle([cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2], outline=colors[int(c)], width=2)
        draw.text((cx - w / 2, cy - h / 2 - 10), names[int(c)], fill=colors[int(c)])
    ax.imshow(img); ax.set_title(image_id); ax.axis('off')
plt.tight_layout()